# MNIST NumPy MLP — 원본 핵심 코드만 정리

원본 `mnist128_64.ipynb`와 `mnist256_128.ipynb`에서 **데이터 준비 → 모델 정의 → 학습 → 테스트 정확도 평가**에 필요한 부분만 남겼습니다. 기존 Google Colab의 `train.csv` 경로와 학습 설정을 그대로 사용합니다. 새로운 저장 기능, INT8 연산, 성능·메모리 벤치마크는 추가하지 않았습니다.

**실행:** 위에서 아래로 순서대로 실행합니다. ① 128→64 모델 학습/평가 후 ② 256→128 모델 학습/평가가 진행됩니다. 원본과 같이 두 모델은 각각 데이터를 무작위로 섞어 70:30 분할하며, 두 번째 모델을 시작하면 전역 가중치 변수는 두 번째 모델 값으로 바뀝니다.

## 공통: 라이브러리와 데이터 로드

In [1]:
%pip install "cupy-cuda13x[ctk]"
%pip install "pandas"

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import cupy as np
import pandas as pd

train = pd.read_csv(r'C:\Users\User\Desktop\mnist-quantization\train.csv')

## 공통: 활성화·순전파·손실·역전파 (원본 코드)

In [3]:
def ReLU(X):
    return np.maximum(0,X)

def softmax(X):
    return np.exp(X-np.max(X,axis=1,keepdims=True))/np.sum(np.exp(X-np.max(X,axis=1,keepdims=True)),axis=1,keepdims=True)

def forward(X):
    global W1,W2,W3,b1,b2,b3
    dic = {}
    r = X.copy()
    for j,i in enumerate([[W1,b1], [W2,b2]]):
        r = np.dot(r, i[0]) + i[1]
        dic[f'Z{j+1}'] = r
        r = ReLU(r)
        dic[f'A{j+1}'] = r
    r = np.dot(r,W3) + b3
    dic['Z3'] = r
    dic['Yhat'] = softmax(r)
    return dic

def loss(X,Y):
    return -np.sum(Y * np.log(X))/X.shape[0]

def backward(X,Y):
    global W1,W2,W3,b1,b2,b3
    forwarddict = forward(X)

    N = X.shape[0]
    Z1 = forwarddict['Z1']
    Z2 = forwarddict['Z2']
    Z3 = forwarddict['Z3']
    A1 = forwarddict['A1']
    A2 = forwarddict['A2']
    Yhat = forwarddict['Yhat']

    dZ3 = Yhat - Y
    dW3 = (np.dot(A2.T,dZ3))/N
    db3 = np.mean(dZ3,axis=0)

    dA2 = np.dot(dZ3,W3.T)
    dZ2 = dA2 * (Z2>0)
    dW2 = (np.dot(A1.T,dZ2))/N
    db2 = np.mean(dZ2,axis=0)

    dA1 = np.dot(dZ2,W2.T)
    dZ1 = dA1 * (Z1>0)
    dW1 = (np.dot(X.T,dZ1))/N
    db1 = np.mean(dZ1,axis=0)

    Wn = [W1,W2,W3]
    bn = [b1,b2,b3]
    dn = [dW1,dW2,dW3]
    dbn = [db1,db2,db3]
    for i in range(3):
        Wn[i] -= 0.01*dn[i]
        bn[i] -= 0.01*dbn[i]
    return loss(Yhat,Y)

## 모델 1 — 784 → 128 → 64 → 10

In [4]:
def randweight():
    global W1,W2,W3,b1,b2,b3
    W1 = np.array([[(np.random.rand()*2-1)*0.01 for i in range(128)] for i in range(784)])
    W2 = np.array([[(np.random.rand()*2-1)*0.01 for i in range(64)] for i in range(128)])
    W3 = np.array([[(np.random.rand()*2-1)*0.01 for i in range(10)] for i in range(64)])

    b1 = np.array([(np.random.rand()*2-1)*0.01 for i in range(128)])
    b2 = np.array([(np.random.rand()*2-1)*0.01 for i in range(64)])
    b3 = np.array([(np.random.rand()*2-1)*0.01 for i in range(10)])

In [5]:
randweight()
trainrand = train.sample(frac=1).reset_index(drop=True)
train3 = trainrand[:(trainrand.shape[0]*7//10)].reset_index(drop=True)
test3 = trainrand[(trainrand.shape[0]*7//10):].reset_index(drop=True)
Xtrain3 = train3[[f'pixel{i}' for i in range(train.shape[1]-1)]]
Ytrain2_5 = train3['label']
Xtrain3 = Xtrain3/255
Xtrain3 = Xtrain3.to_numpy()
Ytrain3 = [[0]*10 for i in range(Xtrain3.shape[0])]

In [6]:
for i in range(Xtrain3.shape[0]):
    Ytrain3[i][Ytrain2_5[i]] = 1
Ytrain3 = np.array(Ytrain3)

for i in range(1000):
    for j in range(Xtrain3.shape[0]//128):
        a = backward(Xtrain3[128*j:128*(j+1)],Ytrain3[128*j:128*(j+1)])
    if i%10 == 0:
        print(f'{i}: {a}')
print(f'loss: {a}')

TypeError: Implicit conversion to a NumPy array is not allowed. Please use `.get()` to construct a NumPy array explicitly.

In [ ]:
Xtest3 = test3[[f'pixel{i}' for i in range(train.shape[1]-1)]]
Ytest2_5 = test3['label']
Xtest3 = Xtest3/255
Xtest3 = Xtest3.to_numpy()
Ytest3 = [[0]*10 for i in range(Xtest3.shape[0])]

l = []
forwarddict_test = forward(Xtest3)
for i in range(Xtest3.shape[0]):
    if np.argmax(forwarddict_test["Yhat"][i]) != Ytest2_5[i]:
        l.append(i)

print('===== test ====')
print(f'classified: {Xtest3.shape[0]-len(l)}')
print(f'misclassified: {len(l)}')
print(f'accuracy: {1-len(l)/Xtest3.shape[0]}')

AttributeError: 'numpy.ndarray' object has no attribute 'to_numpy'

## 모델 2 — 784 → 256 → 128 → 10

원본과 동일하게 `randweight()`를 새 모델 크기로 재정의하고 새 가중치를 초기화합니다. 공통 `forward`, `backward` 함수는 그대로 사용합니다.

In [ ]:
def randweight():
    global W1,W2,W3,b1,b2,b3
    W1 = np.array([[(np.random.rand()*2-1)*0.01 for i in range(256)] for i in range(784)])
    W2 = np.array([[(np.random.rand()*2-1)*0.01 for i in range(128)] for i in range(256)])
    W3 = np.array([[(np.random.rand()*2-1)*0.01 for i in range(10)] for i in range(128)])

    b1 = np.array([(np.random.rand()*2-1)*0.01 for i in range(256)])
    b2 = np.array([(np.random.rand()*2-1)*0.01 for i in range(128)])
    b3 = np.array([(np.random.rand()*2-1)*0.01 for i in range(10)])

In [ ]:
randweight()
trainrand = train.sample(frac=1).reset_index(drop=True)
train3 = trainrand[:(trainrand.shape[0]*7//10)].reset_index(drop=True)
test3 = trainrand[(trainrand.shape[0]*7//10):].reset_index(drop=True)
Xtrain3 = train3[[f'pixel{i}' for i in range(train.shape[1]-1)]]
Ytrain2_5 = train3['label']
Xtrain3 = Xtrain3/255
Xtrain3 = Xtrain3.to_numpy()
Ytrain3 = [[0]*10 for i in range(Xtrain3.shape[0])]
for i in range(Xtrain3.shape[0]):
    Ytrain3[i][Ytrain2_5[i]] = 1
Ytrain3 = np.array(Ytrain3)

In [ ]:
for i in range(1000):
    for j in range(Xtrain3.shape[0]//128):
        a = backward(Xtrain3[128*j:128*(j+1)],Ytrain3[128*j:128*(j+1)])
    if i%5 == 0:
        print(f'{i}: {a}')
print(f'loss: {a}')

0: 2.302326293659223
5: 2.3001281545498813
10: 2.2987228858264395
15: 2.286648969376379
20: 1.6617957254635847
25: 0.8911421998108616
30: 0.788765052715091
35: 0.7194016056822775
40: 0.6479295392885582
45: 0.5684395982684389
50: 0.5055872993584651
55: 0.45796954226844955
60: 0.4186807474659566
65: 0.38451239872214416
70: 0.3564546088058174
75: 0.33381616385652146
80: 0.3151046892861649
85: 0.29953390638814126
90: 0.2849352160184706
95: 0.27100814952834096
100: 0.25700423475713685
105: 0.24457331381705047
110: 0.23270034531427855
115: 0.22164274208044582
120: 0.2118642136682624
125: 0.20248287788981964
130: 0.1931235918715299
135: 0.18481785302679948
140: 0.17681466420356862
145: 0.1690184981666515
150: 0.16134811892831408
155: 0.1541394672243482
160: 0.14717193024133243
165: 0.1401974989637234
170: 0.13373774462656157
175: 0.12772344875113023
180: 0.12148720300214702
185: 0.11614339495414915
190: 0.11085251949997013
195: 0.10574802120263792
200: 0.10089732001259238
205: 0.0959987843626

In [ ]:
Xtest3 = test3[[f'pixel{i}' for i in range(train.shape[1]-1)]]
Ytest2_5 = test3['label']
Xtest3 = Xtest3/255
Xtest3 = Xtest3.to_numpy()
Ytest3 = [[0]*10 for i in range(Xtest3.shape[0])]

l = []
forwarddict_test = forward(Xtest3)
for i in range(Xtest3.shape[0]):
    if np.argmax(forwarddict_test["Yhat"][i]) != Ytest2_5[i]:
        l.append(i)

print('===== test ====')
print(f'classified: {Xtest3.shape[0]-len(l)}')
print(f'misclassified: {len(l)}')
print(f'accuracy: {1-len(l)/Xtest3.shape[0]}')

===== test ====
classified: 12238
misclassified: 362
accuracy: 0.9712698412698413
